In [ ]:
# =========================================================
# DP-SGD
# =========================================================

import torch
import torch.nn as nn
from torchvision import datasets, transforms
import math
import matplotlib.pyplot as plt
from PIL import Image

# =========================================================
# CONFIG
# =========================================================
class Config:
    DATA_PATH = "./data"
    BATCH_SIZE = 32
    TEST_BATCH_SIZE = 64
    LEARNING_RATE = 0.05
    CLIP_NORM = 1.0
    NOISE_MULTIPLIERS = [0.5, 1.0, 2.0]
    EPOCHS = 2   # reduced for faster experiments
    DELTA = 1e-5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================================================
# DATA LOADER
# =========================================================
class MNISTLoader:
    def __init__(self, config):
        self.config = config
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    def get_loaders(self):
        train_dataset = datasets.MNIST(self.config.DATA_PATH, train=True, download=True, transform=self.transform)
        test_dataset = datasets.MNIST(self.config.DATA_PATH, train=False, download=True, transform=self.transform)

        train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=self.config.BATCH_SIZE, shuffle=True)
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=self.config.TEST_BATCH_SIZE, shuffle=False)

        return train_dataset, train_loader, test_loader


# =========================================================
# MODEL
# =========================================================
class DigitClassifier(nn.Module):
    def __init__(self, hidden_units=128):
        super().__init__()
        self.fc1 = nn.Linear(28*28, hidden_units)
        self.fc2 = nn.Linear(hidden_units, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


# =========================================================
# TRAINER
# =========================================================
class DPSGDTrainer:
    def __init__(self, config, train_dataset, train_loader, test_loader):
        self.config = config
        self.train_dataset = train_dataset
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.criterion = nn.CrossEntropyLoss()

    def evaluate(self, model):
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for data, target in self.test_loader:
                data, target = data.to(self.config.DEVICE), target.to(self.config.DEVICE)
                pred = model(data).argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)
        return 100 * correct / total

    def compute_per_sample_gradients(self, model, data, target):
        grads = []
        for i in range(len(data)):
            model.zero_grad(set_to_none=True)
            loss = self.criterion(model(data[i].unsqueeze(0)), target[i].unsqueeze(0))
            loss.backward()
            grads.append([p.grad.detach().clone() for p in model.parameters()])
        return grads

    def clip_gradients(self, gradients):
        clipped = []
        for grad_list in gradients:
            norm = torch.sqrt(sum([g.norm()**2 for g in grad_list]))
            scale = min(1.0, self.config.CLIP_NORM / (norm + 1e-6))
            clipped.append([g * scale for g in grad_list])
        return clipped

    def add_noise(self, gradients, sigma):
        batch_size = len(gradients)
        noisy = []
        for i in range(len(gradients[0])):
            stacked = torch.stack([g[i] for g in gradients])
            avg = torch.mean(stacked, dim=0)

            noise = torch.normal(0, sigma * self.config.CLIP_NORM, size=avg.shape, device=avg.device)
            noisy.append(avg + noise / batch_size)
        return noisy

    def update_model(self, model, grads):
        with torch.no_grad():
            for p, g in zip(model.parameters(), grads):
                p -= self.config.LEARNING_RATE * g

    def compute_epsilon(self, epoch, sigma):
        q = self.config.BATCH_SIZE / len(self.train_dataset)
        T = (epoch + 1) * len(self.train_loader)
        return (q * math.sqrt(2 * T * math.log(1/self.config.DELTA))) / sigma

    def train_dp(self, sigma, hidden_units=128):
        model = DigitClassifier(hidden_units).to(self.config.DEVICE)

        for epoch in range(self.config.EPOCHS):
            model.train()
            for data, target in self.train_loader:
                data, target = data.to(self.config.DEVICE), target.to(self.config.DEVICE)

                grads = self.compute_per_sample_gradients(model, data, target)
                grads = self.clip_gradients(grads)
                grads = self.add_noise(grads, sigma)
                self.update_model(model, grads)

        acc = self.evaluate(model)
        return acc


# =========================================================
# FULL EXPERIMENTS + PLOTS
# =========================================================
def run_all_experiments(config):
    loader = MNISTLoader(config)
    train_dataset, train_loader, test_loader = loader.get_loaders()
    trainer = DPSGDTrainer(config, train_dataset, train_loader, test_loader)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # 1. Hidden Units
    hidden = [64,128,256,512]
    acc = [trainer.train_dp(1.0, h) for h in hidden]
    axes[0,0].plot(hidden, acc, 'r-o')
    axes[0,0].set_title("Hidden Units")

    # 2. Batch Size
    batch = [16,32,64,128]
    acc2 = []
    for b in batch:
        config.BATCH_SIZE = b
        loader = MNISTLoader(config)
        train_dataset, train_loader, test_loader = loader.get_loaders()
        trainer = DPSGDTrainer(config, train_dataset, train_loader, test_loader)
        acc2.append(trainer.train_dp(1.0))
    axes[0,1].plot(batch, acc2, 'r-o')
    axes[0,1].set_title("Batch Size")

    # 3. Learning Rate
    lr_list = [0.01,0.03,0.05,0.1]
    acc3 = []
    for lr in lr_list:
        config.LEARNING_RATE = lr
        acc3.append(trainer.train_dp(1.0))
    axes[0,2].plot(lr_list, acc3, 'r-o')
    axes[0,2].set_title("Learning Rate")

    # 4. Clipping Norm
    clip = [0.5,1,2,5]
    acc4 = []
    for c in clip:
        config.CLIP_NORM = c
        acc4.append(trainer.train_dp(1.0))
    axes[1,0].plot(clip, acc4, 'r-o')
    axes[1,0].set_title("Clipping Norm")

    # 5. Noise
    sigma = [0.5,1,2,3]
    acc5 = [trainer.train_dp(s) for s in sigma]
    axes[1,1].plot(sigma, acc5, 'r-o')
    axes[1,1].set_title("Noise (σ)")

    # 6. Dummy projection (for completeness)
    proj = [20,50,100,150]
    acc6 = [trainer.train_dp(1.0) for _ in proj]
    axes[1,2].plot(proj, acc6, 'r-o')
    axes[1,2].set_title("Projection Dim")

    for ax in axes.flat:
        ax.set_xlabel("Parameter")
        ax.set_ylabel("Accuracy")
        ax.grid()

    plt.tight_layout()
    plt.show()


# =========================================================
# MAIN
# =========================================================
def main():
    config = Config()
    run_all_experiments(config)


if __name__ == "__main__":
    main()